In [ ]:
import kaggle_benchmarks as kbench
import json
import re
import math
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end+1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, target, rel_tol=0.015):
    def parse_val(t):
        s = str(t).replace(" ", "").replace(",", "").lower()
        s = re.sub(r"\\times10\^?{?(-?\d+)}?", r"e\1", s)
        m = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if m: 
            try: return float(m.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_val(answer_text)
    if pred is None: return False
    try:
        t_val = float(target)
        if t_val == 0: return abs(pred) < 1e-9
        return math.isclose(pred, t_val, rel_tol=rel_tol)
    except: return False

def classify_failure_fp_11(answer_text):
    if not answer_text or len(str(answer_text)) < 2: return "hallucination"
    return "calculation_error" # Default for numerical tasks

def build_trace(*, task_id, llm, prompt, response, parsed, final_answer, passed, failure_mode):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt
    }

# ----------------------------
# Finite Chain Detachment Expectation
# ----------------------------
@kbench.task(name="FP-11 Finite Chain Detachment Expectation", description="Physics")
def fp_11_finite_chain_detachment_expectation(llm) -> tuple[int, int]:
    prompt = r"""A polymer is confined to a narrow planar trench and follows a prescribed right-angle “staircase” of N=20 unit segments (d=1.00). Its rightmost endpoint is fixed at P_0 = (8.00, 0.00). Moving inward along the chain from this end, the segment directions alternate between a unit step in -x and a unit step in +y (starting with -x for the first segment). Segments can detach only in order from the right end, so at any instant the set of detached segments is a single contiguous block of length n in {0,1,...,20}. Detaching a segment costs energy epsilon = 1.70. Each detached segment has g=3 internal microstates, whereas each attached segment has one. The two free ends are joined by a frictionless ring connected to an ideal force source that maintains a constant tension of magnitude f=0.70 along the direction of the taut free polymer. The ring’s location is fixed at a clamp point A = (15.00, 0.00). For a given n, the free polymer between A and the peel front is taut and takes the shortest possible path in the plane from A to the peel front that does not enter a smooth circular exclusion region (a post) of radius R = 1.20 centered at (10.00, 3.50). What is the expected number of detached segments <n>, in thermal equilibrium for this finite chain?

Return JSON only in the following format:
{
  "final_answer": "<numeric value>"
}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    passed_checks = 0
    final_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        if numeric_pass(final_answer, 4.84):
            passed_checks = 1
        else:
            failure_mode = classify_failure_fp_11(final_answer)

    trace = build_trace(
        task_id="fp_11",
        llm=llm, prompt=prompt, response=response, parsed=parsed, 
        final_answer=final_answer, passed=(passed_checks == 1), failure_mode=failure_mode
    )
    TRACE_LOG.append(trace)
    return (passed_checks, 1)


In [ ]:
fp_11_finite_chain_detachment_expectation.run(kbench.llm)


In [ ]:
results = fp_11_finite_chain_detachment_expectation.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd
trace_df = pd.DataFrame(TRACE_LOG)
trace_df

In [ ]:
trace_df["failure_mode"].value_counts(dropna=False)

In [ ]:
trace_df.to_csv("fp_11_trace_log.csv", index=False)